# Lección 5.5 — Máquinas de vectores de soporte (SVM): la calle más ancha

> **Módulo 5 · Modelos clásicos supervisados** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 5.2 (regresión logística: frontera lineal), Lección 5.3 (escalado y pipelines), Lección 4.4 (sobreajuste vs. subajuste)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué es el **margen** y qué son los **vectores de soporte**.
- Describir **cómo aprende** una SVM: la calle más ancha posible, con cierta tolerancia a errores.
- Explicar la intuición del **truco del kernel** para separar datos que una recta no puede.
- Predecir el efecto de sus hiperparámetros (`C`, `kernel`, `gamma`).
- Reconocer cuándo una SVM es una buena elección y cuándo no.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import recall_score, confusion_matrix
from sklearn.datasets import make_blobs, make_moons, make_circles, load_breast_cancer, load_digits

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: construir la calle más ancha

Una ciudad tiene dos barrios: el **rojo** y el **azul**. El urbanista tiene que trazar una **calle recta** que los separe. Hay muchas calles posibles. ¿Cuál elige?

```
   Calle A (angosta)                  Calle B (ancha)
  🔴 🔴   ║ 🔵                      🔴 🔴  ║     ║  🔵
   🔴 🔴 ║🔵 🔵                      🔴 🔴 ║     ║ 🔵 🔵
  🔴   🔴║ 🔵                        🔴   🔴║     ║  🔵
         ║  🔵 🔵                          ║     ║   🔵 🔵
   (casi toca las casas)             (deja espacio a ambos lados)
```

Las dos separan bien a los vecinos actuales. Pero cuando lleguen **casas nuevas** cerca de la frontera, la calle ancha tiene más espacio de seguridad.

**Una SVM (*Support Vector Machine*) busca exactamente eso: la calle más ancha posible entre las dos clases.** El ancho de la calle se llama **margen**.

### ✋ Ejercicio 1 — Calentamiento
1. ¿Por qué una calle ancha debería equivocarse **menos** con casas nuevas?
2. Imagina que mueves una casa roja que está **muy lejos** de la calle (en el fondo del barrio). ¿Cambia la mejor calle? ¿Y si mueves una casa que está **justo en la orilla**?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Una casa nueva que llegue un poco "desviada" de su barrio todavía caerá del lado correcto. Una calle angosta, pegada a las casas, se equivoca con el menor desvío. Es un modelo más **robusto**: generaliza mejor.
2. La casa lejana **no cambia nada**: la calle no depende de ella. La casa de la orilla **sí**: si se mueve, la calle tiene que moverse. Esas casas de la orilla son las que "sostienen" la calle: se llaman **vectores de soporte**. ¡De ahí el nombre del modelo!
</details>

---
## 2. 📖 El concepto: margen y vectores de soporte

| Término | Qué es | En la analogía |
|---|---|---|
| **Frontera de decisión** | La línea (o plano) que separa las clases | El centro de la calle |
| **Margen** | La distancia entre la frontera y los puntos más cercanos | El ancho de la calle |
| **Vectores de soporte** | Los puntos que quedan justo en la orilla (o dentro de la calle) | Las casas de la orilla |

Lo sorprendente: **la frontera depende solo de los vectores de soporte**. Puedes borrar todos los demás puntos y la SVM encontraría exactamente la misma calle.

Veámoslo con dos grupos bien separados.

In [ ]:
def dibujar_svm(modelo, X, y, ax, titulo, calle=True):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 250),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 250))
    zz = modelo.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz > 0, alpha=.25, cmap="coolwarm")
    ax.contour(xx, yy, zz, levels=[-1, 0, 1] if calle else [0], colors="k",
               linestyles=["--", "-", "--"] if calle else ["-"])
    ax.scatter(*X.T, c=y, cmap="coolwarm", edgecolor="k", s=20)
    if calle:
        sv = modelo.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=120, facecolors="none", edgecolors="lime", linewidths=2)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

X_b, y_b = make_blobs(n_samples=60, centers=2, cluster_std=1.0, random_state=6)
svm_lineal = SVC(kernel="linear", C=1000).fit(X_b, y_b)

fig, ax = plt.subplots(figsize=(5, 4))
dibujar_svm(svm_lineal, X_b, y_b, ax, "Línea continua = frontera · discontinuas = orillas\n◯ verde = vectores de soporte")
plt.show()
print(f"Puntos: {len(X_b)}  ·  Vectores de soporte: {len(svm_lineal.support_vectors_)}")

### 🔮 Ejercicio 2 — ¿Quién sostiene la calle?
Vamos a hacer dos experimentos:
- **A:** borrar 20 puntos que **no** son vectores de soporte y volver a entrenar.
- **B:** borrar **un** vector de soporte y volver a entrenar.

¿En cuál cambiará la frontera?

✍️ *Mi predicción:*

In [ ]:
no_soporte = np.setdiff1d(np.arange(len(X_b)), svm_lineal.support_)
quitar_A = no_soporte[:20]                    # 20 puntos que NO son de soporte
quitar_B = svm_lineal.support_[:1]            # 1 vector de soporte

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
dibujar_svm(svm_lineal, X_b, y_b, axes[0], "Original")
for ax, quitar, nombre in [(axes[1], quitar_A, "A: sin 20 puntos normales"), (axes[2], quitar_B, "B: sin 1 vector de soporte")]:
    X_, y_ = np.delete(X_b, quitar, axis=0), np.delete(y_b, quitar)
    m = SVC(kernel="linear", C=1000).fit(X_, y_)
    dibujar_svm(m, X_, y_, ax, nombre)
    print(f"{nombre:<28} → pendiente de la frontera: {-m.coef_[0, 0] / m.coef_[0, 1]:.3f}")
print(f"{'Original':<28} → pendiente de la frontera: {-svm_lineal.coef_[0, 0] / svm_lineal.coef_[0, 1]:.3f}")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **A:** la frontera es **idéntica** (misma pendiente): esos 20 puntos no sostenían nada.
- **B:** con un solo punto menos, la calle **cambia** (la pendiente pasa de ≈ −0.02 a ≈ −0.12: gira y se ensancha), porque perdió uno de sus 3 soportes.

Esto es muy distinto de la regresión logística (Lección 5.2), donde **todos** los puntos influyen en la frontera. La SVM se fija solo en los casos difíciles, los de la orilla.
</details>

---
## 3. 📖 Cómo aprende: la calle más ancha… con algunas multas

En la vida real los barrios no están perfectamente separados: hay casas rojas metidas en el barrio azul. Si exigimos una calle sin ninguna casa dentro, puede que **no exista**, o que sea angostísima.

La solución es el **margen suave** (*soft margin*): permitimos que algunas casas queden dentro de la calle o del lado equivocado, pero **cada violación paga una multa**. La SVM busca el mejor equilibrio entre:
1. una calle **ancha** (generaliza mejor), y
2. **pocas multas** (pocos puntos mal colocados).

El hiperparámetro **`C`** decide cuánto cuesta cada multa:
- **`C` grande** → multas carísimas → calle **angosta** que intenta no dejar a nadie mal → riesgo de **sobreajuste**.
- **`C` pequeño** → multas baratas → calle **ancha** que tolera errores → riesgo de **subajuste**.

> 💡 Es el mismo `C` de la regresión logística (Lección 5.2): **inverso de la regularización**. `C` grande = poca regularización.

> 💡 **¿Y el descenso de gradiente?** Por dentro, scikit-learn resuelve un problema de optimización con un algoritmo especializado (no con épocas ni tasa de aprendizaje). Lo importante para ti: el resultado es **único** (no hay mínimos locales).

### 🔮 Ejercicio 3 — El precio de las multas
Con dos grupos que **se traslapan**, entrenamos una SVM lineal con `C` = 0.01, 1 y 100.
1. ¿Cuál tendrá la calle más **ancha**?
2. ¿Cuál tendrá **más** vectores de soporte?

✍️ *Mis predicciones:*

In [ ]:
X_t, y_t = make_blobs(n_samples=100, centers=2, cluster_std=2.2, random_state=6)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, c in zip(axes, [0.01, 1, 100]):        # ✏️ prueba otros valores
    m = SVC(kernel="linear", C=c).fit(X_t, y_t)
    ancho = 2 / np.linalg.norm(m.coef_)
    dibujar_svm(m, X_t, y_t, ax, f"C={c}\nancho de la calle {ancho:.2f} · soportes {len(m.support_)} · acierto {m.score(X_t, y_t):.0%}")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. **`C=0.01`** tiene la calle más ancha (≈5.1, contra ≈1.6 con `C=1` y ≈1.0 con `C=100`): las multas son tan baratas que no le importa dejar muchos puntos dentro de la calle.
2. También **`C=0.01`** tiene más vectores de soporte (37, contra 14 y 12): **todo** punto dentro de la calle (o del lado equivocado) cuenta como vector de soporte.

Al subir `C`, la calle se angosta y quedan menos soportes. El acierto en entrenamiento casi no cambia (95–96 %). Aquí la frontera casi no cambia de lugar entre `C=1` y `C=100` porque los datos son casi lineales; con fronteras curvas (Sección 5) verás que un `C` muy grande sí sobreajusta.
</details>

---
## 4. 📖 ¿Qué pasa si una recta no basta? El truco del kernel

En la Lección 5.2 viste que con dos círculos concéntricos una frontera recta acierta como una moneda. Allí lo arreglamos **inventando características** ($x_1^2$, $x_2^2$). La SVM lleva esa idea al extremo.

### La idea en una dimensión
Imagina puntos sobre una línea: los **rojos** en el centro y los **azules** a los lados.

```
   🔵 🔵 🔵   🔴 🔴 🔴 🔴   🔵 🔵 🔵
  ─────────────────────────────────────  x
```

Ningún **punto de corte** los separa (necesitarías dos). Pero si agregas una segunda característica, $x^2$, los azules "suben" y los rojos se quedan abajo… ¡y ahora una línea horizontal los separa!

### 🔮 Ejercicio 4 — Levanta los puntos
Antes de ejecutar: si dibujamos cada punto en $(x, x^2)$, ¿dónde quedarán los rojos y dónde los azules?

✍️ *Mi predicción:*

In [ ]:
x = np.linspace(-3, 3, 25)
clase = (np.abs(x) > 1.5).astype(int)          # 1 = azul (a los lados), 0 = rojo (en el centro)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(x, np.zeros_like(x), c=clase, cmap="coolwarm", edgecolor="k", s=50)
axes[0].set_title("Original: 1 característica (x)\n¿un solo corte los separa?"); axes[0].set_yticks([])
axes[1].scatter(x, x ** 2, c=clase, cmap="coolwarm", edgecolor="k", s=50)
axes[1].axhline(1.5 ** 2, ls="--", c="k")
axes[1].set_title("Con una característica nueva (x²)\nuna línea horizontal basta"); axes[1].set_xlabel("x"); axes[1].set_ylabel("x²")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Los **azules** (|x| grande) quedan **arriba**, porque $x^2$ es grande; los **rojos** (cerca de 0) quedan **abajo**. Una línea horizontal en $x^2 = 2.25$ los separa perfectamente.

Lo que en el espacio original necesitaba una frontera "rara", en un espacio con más dimensiones es una simple recta (o plano).
</details>

### 📖 El "truco": sin crear las características de verdad
Crear a mano $x^2$, $x_1 x_2$, $x_1^3$… se vuelve imposible con muchas características. El **truco del kernel** permite que la SVM trabaje **como si** hubiera creado muchísimas características nuevas (¡incluso infinitas!) sin calcularlas nunca. Solo necesita saber **qué tan parecidos** son dos puntos, y eso lo calcula una función llamada **kernel**.

| `kernel` | Intuición | Frontera |
|---|---|---|
| `"linear"` | Sin características nuevas | Recta / plano |
| `"poly"` | Como añadir potencias y productos (grado `degree`) | Curvas polinómicas |
| `"rbf"` (por defecto) | Cada vector de soporte emite una "campana" de influencia a su alrededor | Cualquier forma: islas, curvas, círculos |

El kernel **RBF** (*radial basis function*) es el más usado. Piensa en cada vector de soporte como una **lámpara** que ilumina su zona con el color de su clase. Un punto nuevo toma el color de la luz que más le llega.

In [ ]:
X_l, y_l = make_moons(n_samples=300, noise=0.2, random_state=0)
X_c, y_c = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=0)

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for fila, (X_, y_, nombre) in enumerate([(X_l, y_l, "Lunas"), (X_c, y_c, "Círculos")]):
    for col, kernel in enumerate(["linear", "rbf"]):
        m = SVC(kernel=kernel).fit(X_, y_)
        cv = cross_val_score(SVC(kernel=kernel), X_, y_, cv=5).mean()
        dibujar_svm(m, X_, y_, axes[fila, col], f"{nombre} · kernel='{kernel}' · CV {cv:.0%}", calle=False)
plt.tight_layout(); plt.show()

### ✋ Ejercicio 5 — Compara
1. ¿Qué exactitud tiene el kernel lineal en los círculos? ¿Te recuerda a algún modelo anterior?
2. ¿Qué kernel usarías si no sabes qué forma tiene la frontera?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. ≈56 %, casi una moneda al aire: como la regresión logística de la Lección 5.2, porque los dos son **clasificadores lineales**. Con `rbf`, la SVM llega a ≈100 % en los círculos y ≈98 % en las lunas (contra ≈84 % del lineal) sin que inventáramos ninguna característica.
2. `rbf` es el punto de partida habitual (por eso es el valor por defecto). Pero compáralo siempre con `linear` usando validación cruzada: si los datos son casi lineales o tienen **muchísimas** características (como el texto), el lineal suele ganar y es mucho más rápido.
</details>

---
## 5. 📖 Hiperparámetros: `C` y `gamma`, la pareja inseparable

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `C` | Precio de cada multa (inverso de la regularización) | Calle angosta, intenta acertar todo → **sobreajuste** | Calle ancha, tolera errores → **subajuste** | 1; probar 0.01–1000 en escala logarítmica |
| `kernel` | Forma de la frontera | — | — | `"rbf"`; comparar con `"linear"` |
| `gamma` (rbf, poly) | **Alcance** de cada punto: qué tan rápido se apaga su "lámpara" | Lámparas de alcance corto → islas alrededor de cada punto → **sobreajuste** | Lámparas de alcance largo → frontera muy suave → **subajuste** | `"scale"` (por defecto); probar 0.001–10 |
| `degree` (poly) | Grado del polinomio | Más curvas | Más simple | 3 |
| `class_weight` | Multas distintas por clase | — | — | `None`, o `"balanced"` si hay desbalance |

> 🧠 **Regla mnemotécnica:** en una SVM, **subir `C` o subir `gamma` complica el modelo**. Por eso se ajustan **juntos**: un `gamma` alto con un `C` alto es la receta del sobreajuste.

### 🔮 Ejercicio 6 — El alcance de las lámparas
Entrenamos una SVM RBF en las lunas con `gamma` = 0.1, 1 y 100. ¿Cuál dibujará islas alrededor de cada punto? ¿Cuál será casi una recta?

✍️ *Mis predicciones:*

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(*make_moons(n_samples=400, noise=0.35, random_state=1), test_size=0.3, random_state=1)

def comparar(parametro, valores, **fijos):
    fig, axes = plt.subplots(1, len(valores), figsize=(4 * len(valores), 3.8))
    for ax, v in zip(axes, valores):
        m = SVC(**{parametro: v}, **fijos).fit(X_tr, y_tr)
        dibujar_svm(m, X_tr, y_tr, ax,
                    f"{parametro}={v}\ntrain {m.score(X_tr, y_tr):.0%} · test {m.score(X_te, y_te):.0%}", calle=False)
    plt.tight_layout(); plt.show()

comparar("gamma", [0.1, 1, 100])

<details><summary><b>👀 Ver respuesta</b></summary>

- **`gamma=0.1`** → lámparas de alcance enorme: la frontera es muy suave y apenas se curva (83 % en entrenamiento) → tiende al **subajuste**.
- **`gamma=1`** → sigue la forma de las lunas (88 % en entrenamiento). En prueba empata con 0.1 (≈86 %): con estos datos tan ruidosos, ambas son razonables.
- **`gamma=100`** → cada punto ilumina solo su alrededor inmediato: **islas** alrededor de puntos sueltos, 96 % en entrenamiento y solo 77 % en prueba → **sobreajuste**.

¿Te suena? Es el mismo patrón que `n_neighbors` en KNN (Lección 5.3), pero al revés: en KNN, un k **pequeño** sobreajusta; aquí, un `gamma` **grande**.
</details>

### 🔮 Ejercicio 7 — Ahora `C`, con fronteras curvas
Mismo kernel RBF (`gamma` por defecto), con `C` = 0.01, 1 y 1000. ¿Qué esperas?

✍️ *Mis predicciones:*

In [ ]:
comparar("C", [0.01, 1, 1000])

<details><summary><b>👀 Ver respuesta</b></summary>

- **`C=0.01`** → multas casi gratis: al modelo casi no le importa equivocarse y prácticamente pinta todo de un color → **subajuste** extremo (54 % en entrenamiento, 41 % en prueba).
- **`C=1`** → equilibrio (88 % / 87 %).
- **`C=1000`** → multas carísimas: la frontera se retuerce para acertar puntos ruidosos → sube en entrenamiento (91 %) y **baja en prueba** (83 %): **sobreajuste**.
</details>

### ✏️ Ejercicio 8 — Buscar `C` y `gamma` juntos
Como las dos perillas interactúan, se prueban **todas las combinaciones** con validación cruzada. `GridSearchCV` lo hace por ti (lo verás a fondo en la Lección 7.3). El mapa de calor muestra la exactitud de cada combinación.

In [ ]:
VALORES_C = [0.01, 0.1, 1, 10, 100, 1000]          # ✏️
VALORES_GAMMA = [0.01, 0.1, 1, 10, 100]            # ✏️
busqueda = GridSearchCV(SVC(), {"C": VALORES_C, "gamma": VALORES_GAMMA}, cv=5).fit(X_tr, y_tr)

tabla = pd.DataFrame(busqueda.cv_results_["mean_test_score"].reshape(len(VALORES_C), len(VALORES_GAMMA)),
                     index=[f"C={c}" for c in VALORES_C], columns=[f"γ={g}" for g in VALORES_GAMMA])
plt.imshow(tabla, cmap="viridis"); plt.colorbar(label="Exactitud CV")
plt.xticks(range(len(VALORES_GAMMA)), tabla.columns); plt.yticks(range(len(VALORES_C)), tabla.index)
for i in range(len(VALORES_C)):
    for j in range(len(VALORES_GAMMA)):
        plt.text(j, i, f"{tabla.iloc[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
plt.title("Exactitud en validación cruzada"); plt.show()
print("Mejor combinación:", busqueda.best_params_, f"→ {busqueda.best_score_:.1%}")

Responde:
1. ¿En qué zona del mapa está el sobreajuste? ¿Y el subajuste?
2. ¿Por qué la zona buena forma una especie de **diagonal**?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Subajuste:** la fila `C=0.01` (≈0.54 en toda la fila) y la esquina de `C` y `gamma` pequeños. **Sobreajuste:** la columna `gamma=100` (≈0.79) y la esquina de `C` y `gamma` grandes a la vez (`C=1000`, `γ=10`: 0.80).
2. Porque las dos perillas se compensan: un `gamma` alto (más complejo) funciona con un `C` bajo (`C=1`, `γ=10`: 0.89), y un `gamma` bajo necesita un `C` alto (`C=1000`, `γ=0.1`: 0.89). Por eso nunca se ajusta una sin la otra. Varias combinaciones empatan alrededor de 0.89; `GridSearchCV` se queda con la primera que encuentra.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Exactitud sorprendentemente baja | Datos **sin escalar**: el kernel RBF usa distancias, como KNN | `make_pipeline(StandardScaler(), SVC())` |
| 100 % en entrenamiento y mucho menos en prueba | `C` y/o `gamma` demasiado altos | Bajarlos; buscarlos juntos con `GridSearchCV` |
| Siempre predice la clase mayoritaria | Clases desbalanceadas | `class_weight="balanced"` |
| Entrenar tarda muchísimo (horas) | `SVC` crece muy mal con muchas filas (más que proporcionalmente) | Con más de ~50,000 filas: `LinearSVC`, submuestrear u otro modelo |
| Pides `predict_proba` y da error | `SVC` no calcula probabilidades por defecto | Usar `decision_function` (distancia a la calle) o calibrar con `CalibratedClassifierCV` |

### 🔮 Ejercicio 9 — ¿Escalar importa tanto como en KNN?
Volvemos a los tumores de mama (1 = maligno). ¿Cuánta diferencia esperas entre una SVM RBF sin escalar y una con escalado?

✍️ *Mi apuesta:*

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
yc = 1 - yc     # 1 = maligno
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=0)

print(f"SVM sin escalar: {cross_val_score(SVC(), Xc_tr, yc_tr, cv=5).mean():.1%}")
print(f"SVM con escalar: {cross_val_score(make_pipeline(StandardScaler(), SVC()), Xc_tr, yc_tr, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sin escalar ≈ 91 %; con escalar ≈ 97 %. El kernel RBF mide qué tan **cerca** están dos puntos, así que, igual que KNN, una columna con números grandes (el área, hasta 2,500) aplasta a las demás. **Regla: SVM siempre dentro de un pipeline con escalado.**
</details>

### 📖 ¿Y las probabilidades? `decision_function`
La regresión logística da probabilidades; la SVM da algo distinto: **a qué distancia de la calle** está cada punto (`decision_function`), con signo según el lado.
- Positivo grande → muy dentro del lado "1".
- Cerca de 0 → en plena calle: caso dudoso.
- Negativo grande → muy dentro del lado "0".

No es una probabilidad, pero **sí ordena** los casos del más al menos sospechoso, así que puedes **mover el umbral** igual que en la Lección 5.2: en lugar de cortar en 0, cortas en otro valor.

> 📝 Si necesitas probabilidades de verdad (por ejemplo, para calcular un umbral a partir de costos), envuelve la SVM con `CalibratedClassifierCV(SVC(), ensemble=False)`. En versiones antiguas de scikit-learn se usaba `SVC(probability=True)`, que está obsoleto desde la 1.9.

### ✋ Ejercicio 10 — Mover el umbral de una SVM
Por defecto la SVM marca "maligno" si `decision_function` > 0. Ejecuta y responde: ¿qué umbral usarías si dejar pasar un tumor maligno fuera muchísimo peor que una falsa alarma?

In [ ]:
svm_med = make_pipeline(StandardScaler(), SVC()).fit(Xc_tr, yc_tr)
distancia = svm_med.decision_function(Xc_te)

for umbral in [0, -0.5, -1]:          # ✏️ prueba otros
    pred = (distancia > umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(yc_te, pred).ravel()
    print(f"umbral {umbral:>4} → malignos detectados {tp}/{tp + fn} · falsas alarmas {fp}")

<details><summary><b>👀 Ver respuesta</b></summary>

Bajando el umbral de 0 a −1, la SVM pasa de detectar 50 a detectar **los 53** malignos, a cambio de muchas más falsas alarmas (de 4 a 24). Con −0.5 queda un punto intermedio (52 de 53, 12 falsas alarmas). Con un falso negativo muy caro, un umbral negativo como −0.5 o −1 tiene sentido. **El umbral se elige con validación cruzada en el entrenamiento** (Lección 5.2), no mirando la prueba como hicimos aquí para ilustrar.
</details>

---
## 7. 📋 Ficha del modelo

| | Máquina de vectores de soporte (SVM) |
|---|---|
| **Tipo** | Supervisado: clasificación (`SVC`) y regresión (`SVR`) |
| **Intuición** | Traza la calle más ancha posible entre las clases; solo los puntos de la orilla (vectores de soporte) la definen |
| **Cómo aprende** | Maximiza el margen pagando una multa (`C`) por cada punto mal colocado; con un *kernel* trabaja como si tuviera muchas características nuevas |
| **Hiperparámetros clave** | `C`, `kernel`, `gamma` (y `class_weight`) |
| **✅ Fortalezas** | Muy precisa con datos medianos · Fronteras de cualquier forma (kernel RBF) · Funciona bien con muchas características (texto, con kernel lineal) · Resultado único, sin mínimos locales |
| **❌ Debilidades** | **Necesita escalar** · Lenta con muchas filas (más de ~50,000) · `C` y `gamma` hay que ajustarlos juntos · Poco interpretable con kernel RBF · Sin probabilidades directas |
| **Úsalo cuando…** | Tienes hasta decenas de miles de filas, datos numéricos escalables y buscas precisión (imágenes pequeñas, texto, señales) |
| **Evítalo cuando…** | Tienes millones de filas, necesitas explicar cada decisión o necesitas probabilidades calibradas sin pasos extra |

## 8. 🧠 Resumen en 5 puntos
1. Una SVM busca la **calle más ancha** (margen máximo) entre las clases.
2. Solo los **vectores de soporte**, los puntos de la orilla, definen la frontera.
3. `C` es el precio de cada multa: **alto → calle angosta y sobreajuste; bajo → calle ancha y subajuste**.
4. El **truco del kernel** (RBF) permite fronteras curvas sin crear características a mano; `gamma` es el alcance de cada punto (**alto → islas y sobreajuste**).
5. Siempre **escalada en un pipeline**, con `C` y `gamma` buscados **juntos** por validación cruzada; da distancias (`decision_function`), no probabilidades.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈35 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué son los vectores de soporte?
- a) Todos los puntos de entrenamiento · b) Los puntos más alejados de la frontera · c) Los puntos en la orilla de la calle o dentro de ella · d) Los coeficientes del modelo

**2.** Tu SVM RBF tiene 100 % en entrenamiento y 70 % en prueba. ¿Qué cambio tiene **más** sentido?
- a) Subir `gamma` · b) Subir `C` · c) Bajar `gamma` y/o `C` · d) Quitar el escalado

**3.** Con `kernel="linear"`, ¿a qué modelo del módulo se parece más una SVM en el tipo de frontera?
- a) KNN · b) Árbol de decisión · c) Regresión logística · d) Ninguno

**4.** Tienes 2 millones de filas y 20 columnas. ¿Qué harías con `SVC(kernel="rbf")`?
- a) Usarlo tal cual · b) Considerar `LinearSVC`, submuestrear o usar otro modelo · c) Subir `gamma` para que sea más rápido · d) No escalar para ahorrar tiempo

**5.** Necesitas probabilidades para decidir con costos. ¿Qué haces con una `SVC`?
- a) `predict_proba` directamente · b) Calibrarla con `CalibratedClassifierCV` (o usar `decision_function` y elegir el umbral con validación cruzada) · c) Subir `C` · d) Imposible

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Las casas de la orilla (y las que violan el margen). Son las únicas que definen la calle.
2. **c)** Brecha enorme = sobreajuste; en SVM, `gamma` y `C` altos complican el modelo.
3. **c)** Las dos trazan una frontera recta; cambian en **cómo** eligen la recta (la logística usa todos los puntos y la entropía cruzada; la SVM, el margen).
4. **b)** El costo de entrenar `SVC` crece mucho más rápido que el número de filas.
5. **b)** `SVC` no da probabilidades por defecto.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Una regresión logística usa **todos** los puntos para definir la frontera; una SVM, solo los de la orilla. ¿Qué ventaja tiene esto si hay muchos puntos "fáciles" lejos de la frontera? ¿Y qué riesgo, si algunos de los puntos de la orilla están **mal etiquetados**?

**B. Caso de análisis 💼** Un equipo necesita clasificar correos en 5 departamentos. Tienen 8,000 correos y, al convertirlos en palabras, 30,000 columnas. Proponen una SVM con kernel RBF y una búsqueda de 8 valores de `C` × 8 de `gamma` con validación cruzada de 5 pliegues. ¿Cuántos modelos entrenarían? ¿Qué propondrías en su lugar como primer intento?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** **Ventaja:** a la SVM no le afectan los puntos fáciles; se concentra en los casos difíciles, y añadir más ejemplos obvios no la "distrae". **Riesgo:** como pocos puntos sostienen la calle, unos pocos errores de etiqueta en la orilla pueden moverla mucho, sobre todo con `C` alto. Un `C` más bajo (multas baratas) la hace más tolerante a esos puntos.

**B.** 8 × 8 × 5 = **320 entrenamientos** de una SVM RBF con 30,000 columnas: lentísimo. Con texto (muchas columnas, datos dispersos), las clases suelen separarse bien con una frontera **lineal**, así que un primer intento sensato es `LinearSVC` (o regresión logística) buscando solo `C`: 8 × 5 = 40 entrenamientos mucho más rápidos. Naive Bayes (Lección 5.6) también es una excelente línea base para texto.
</details>

### 9.3 🧩 Práctica integradora: "Lector de dígitos escritos a mano"

El servicio postal quiere leer automáticamente los dígitos escritos a mano en los códigos postales. Tienes 1,797 imágenes de 8×8 píxeles (64 características, cada una de 0 a 16).

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
digitos = load_digits()
Xd, yd = digitos.data, digitos.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, stratify=yd, random_state=0)

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, img, etiqueta in zip(axes, digitos.images, digitos.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(etiqueta); ax.axis("off")
plt.show()
print(f"Entrenamiento: {len(Xd_tr)} · Prueba: {len(Xd_te)} · Clases: {len(set(yd))}")

**Paso 2 🔮 — Lineal contra RBF.** Antes de ejecutar: ¿crees que una frontera lineal bastará para distinguir 10 dígitos con 64 píxeles? ¿Cuál ganará?

✍️ *Mi apuesta:*

In [ ]:
for kernel in ["linear", "rbf"]:
    acc = cross_val_score(make_pipeline(StandardScaler(), SVC(kernel=kernel)), Xd_tr, yd_tr, cv=5).mean()
    print(f"kernel={kernel:<6} → exactitud CV {acc:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡**Empatan**! Los dos llegan a ≈97.8 %. Con 64 características ya hay "mucho espacio" para que una frontera lineal separe muy bien las clases. Es una lección importante: **no asumas que el kernel complejo siempre gana**; compruébalo.

> 📝 Con 10 clases, `SVC` entrena internamente una SVM por cada **par** de dígitos (45 en total) y combina sus votos. No tienes que hacer nada especial.
</details>

**Paso 3 ✏️ — Busca `C` y `gamma`.** En un pipeline, los hiperparámetros del modelo se nombran con el prefijo `svc__`.

In [ ]:
pipe = make_pipeline(StandardScaler(), SVC())
rejilla = {"svc__C": [0.1, 1, 10, 100],                 # ✏️
           "svc__gamma": [0.0005, 0.005, 0.05, 0.5]}    # ✏️
busqueda_d = GridSearchCV(pipe, rejilla, cv=5).fit(Xd_tr, yd_tr)

print("Mejor combinación:", busqueda_d.best_params_, f"→ exactitud CV {busqueda_d.best_score_:.1%}")
pd.DataFrame(busqueda_d.cv_results_["mean_test_score"].reshape(4, 4),
             index=[f"C={c}" for c in rejilla["svc__C"]],
             columns=[f"γ={g}" for g in rejilla["svc__gamma"]]).round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

La mejor zona tiene un `gamma` pequeño-medio (≈0.005) con un `C` de 10 o más, llegando a ≈98.3 % (la mejor: `C=100`, `γ=0.005`). Con `gamma=0.5` la exactitud se desploma (≈12–34 %): cada imagen solo "ilumina" a sí misma → sobreajuste extremo. Con `gamma=0.0005` y `C=0.1` también es mala (≈25 %): subajuste. Fíjate otra vez en la diagonal: con `γ=0.0005` hace falta un `C` grande.
</details>

**Paso 4 ▶️ — Examen final, una sola vez.** `GridSearchCV` ya reentrenó el mejor modelo con todo el entrenamiento.

In [ ]:
mejor = busqueda_d.best_estimator_
pred_d = mejor.predict(Xd_te)
print(f"Exactitud en prueba: {np.mean(pred_d == yd_te):.1%}  ·  errores: {np.sum(pred_d != yd_te)} de {len(yd_te)}")
print(f"Vectores de soporte: {len(mejor[-1].support_)} de {len(Xd_tr)} imágenes de entrenamiento")

<details><summary><b>👀 Ver respuesta</b></summary>

≈98 % en prueba (8 imágenes mal leídas de 450), coherente con la validación cruzada. Fíjate: la SVM usa como vectores de soporte 546 de las 1,347 imágenes (≈40 %): con 10 clases hay muchas fronteras que sostener, y los dígitos "ambiguos" son los que las definen.
</details>

**Paso 5 ✋ — Mira los errores.** ¿Qué dígitos confunde? ¿Tú los habrías leído bien?

In [ ]:
errores = np.where(pred_d != yd_te)[0]
fig, axes = plt.subplots(1, len(errores), figsize=(1.4 * len(errores), 1.9))
for ax, i in zip(np.atleast_1d(axes), errores):
    ax.imshow(Xd_te[i].reshape(8, 8), cmap="gray_r"); ax.axis("off")
    ax.set_title(f"real {yd_te[i]}\nleyó {pred_d[i]}", fontsize=9)
plt.show()

✍️ *¿Qué observas?*

<details><summary><b>👀 Ver respuesta</b></summary>

Los errores suelen ser dígitos **realmente ambiguos** a 8×8 píxeles: trazos incompletos o deformes que hasta a una persona le costaría leer. Cuando los errores de un modelo se parecen a los errores humanos, es buena señal: el modelo aprendió el patrón y lo que falla son los casos difíciles. Para mejorar más, harían falta más datos o imágenes con más resolución (y modelos como las CNN del Módulo 10).
</details>

**Paso 6 ✋ — Recomendación.** El servicio postal procesa **50 millones** de sobres al día y quiere reentrenar el modelo cada semana con todos los dígitos nuevos. ¿Seguirías usando `SVC` con kernel RBF? ¿Qué le dirías al equipo?

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Para **predecir**, la SVM es rápida (solo compara con los vectores de soporte). El problema es **reentrenar** con millones de imágenes nuevas cada semana: `SVC` con RBF no escala a ese tamaño. Opciones: entrenar con una muestra representativa, usar una versión lineal (`LinearSVC`), o pasar a modelos que escalan mejor (redes neuronales, Módulo 9–10). La SVM sigue siendo excelente como **prototipo** y línea base fuerte con datos medianos.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "Support Vector Machines Part 1 (of 3): Main Ideas!!!" y "The Polynomial and Radial Kernels" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Support Vector Machines*, y el ejemplo *RBF SVM parameters* (mapa de `C` × `gamma`).
- 🎮 **Interactivo:** busca "LIBSVM applet" o "SVM interactive demo": coloca puntos y observa cómo cambian la calle y los vectores de soporte.

---
⬅️ **Anterior:** [5.4 Árboles de decisión: el juego de las 20 preguntas](5.4-arboles-de-decision.ipynb) · ➡️ **Siguiente:** [5.6 Naive Bayes: probabilidad con supuestos ingenuos](5.6-naive-bayes.ipynb)